# Picking a Threshold, For Real — Assignment

**Week 2 · Assignment 2** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026
**Points:** 70 &nbsp;·&nbsp; **Due:** *End of Week 2*

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week2/assignments/week02_threshold_decision.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Continues Lesson 11 ("Evaluation Walkthrough on the Churn Dataset") — same
dataset, same pipeline. This week adds Part B: your first modeling decision
under a real cost tradeoff. No Part C yet — that starts Week 3. You'll
revisit this exact retention-call economics with a stronger model in
Week 5.*

## Setup

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

from gbus564_coursework.utils.course_data import load_churn
from gbus564_coursework.utils.assignment_checks import check_close, check_equal, check_in_range, check_type

## Before You Start

- **Format this week: A + B — 70 pts.** **A. Extend** (40 pts, scaffolded
  fill-ins) → **B. Decide** (30 pts: 20 for the number, 10 for your
  justification). No Part C yet — that starts Week 3, once you've made one
  modeling decision here to build on.
- **Self-checks are feedback, not the grade.** The `assert` cells after each
  Part A fill-in are checkpoints: green output means you're on track.
- **Submission:** *Restart & Run All*, then download the executed `.ipynb`
  and submit that file — outputs must be present in what you turn in.
- **AI use is expected** (see syllabus) — but Part B is graded on judgment
  against *your own* results, so ground your answer in the numbers your own
  run produced.

## Summary of Answers

*Fill this in last, after finishing Part B below.*

| Part | Your Answer |
| --- | --- |
| **B — Decide** | _Your answer here._ |

## 1. Business Problem

Lesson 11 already asked which customers are likely to cancel, and picked a
threshold by targeting 70% recall — a modeling choice, not a business one.
This assignment asks the business version of the same question: given what
a retention call actually costs and what a save is actually worth, what
threshold should Customer Success use?

**Business question:** at what predicted-churn-probability threshold should
Customer Success start making retention calls, given the dollars behind a
call and a save?

**Why it matters:** call every customer the model flags as risky and most
of the calls are wasted on people who'd have stayed anyway; call no one and
preventable churn walks out the door with no attempt to stop it.

## 2. Data — Part A: Extend

*Same dataset, same split as Lesson 11 — given below, nothing to edit yet.*

In [ ]:
df = load_churn()
print(f"positive (churn) rate: {df['Churn'].mean():.1%}")
df.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

preprocess = ColumnTransformer(
    [("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough",
)

## 3. Model — Part A: Extend

*Same kind of candidate search as Lesson 11 — cross-validated `max_depth`,
scored by average precision, is given below — but over a different set of
depths, so Lesson 11's winner doesn't carry over. Your fill-in: pick the best
candidate and fit it.*

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline

def make_model(max_depth):
    clf = RandomForestClassifier(max_depth=max_depth, random_state=RANDOM_SEED)
    return Pipeline([("prep", preprocess), ("clf", clf)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
candidates = [4, 8, 12, 16]
cv_scores = {
    depth: cross_val_score(make_model(depth), X_train, y_train, cv=cv,
                            scoring="average_precision").mean()
    for depth in candidates
}
for depth, score in cv_scores.items():
    print(f"max_depth={depth}:  avg. CV average-precision = {score:.3f}")

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

In [ ]:
# TODO: pick the candidate with the best cv_scores value, then fit it on
# the full training set with make_model().
# YOUR CODE HERE

print(f"Best candidate: max_depth={best_depth}")

In [ ]:
check_type(model, Pipeline, "model is a fitted Pipeline")
check_equal(model.named_steps["clf"].max_depth, best_depth, "model was fit with the best_depth candidate")
check_equal(best_depth in candidates, True, "best_depth came from the candidates you searched")

## 4. Evaluate — Part A: Extend

*Score against the baseline, same as Lesson 11 — given below, along with the
model's confusion matrix at the default 0.5 threshold. Your fill-in: pull the
four counts out of that matrix and compute precision and recall **by hand**.*

**Do not use a package function for this** — no `precision_score`,
`recall_score`, `classification_report`, or anything similar from sklearn or
another library. Compute both with plain arithmetic on the four counts.

- **Precision** — of the customers the model flags, what share actually churn?
- **Recall** — of the customers who actually churn, what share does the model flag?

These same four counts are what Part B turns into dollars: every flagged
customer is a call, and every flagged churner is a save.

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix

baseline_preds = baseline.predict(X_test)
model_proba = model.predict_proba(X_test)[:, 1]
model_preds = model.predict(X_test)

print(f"Baseline accuracy: {accuracy_score(y_test, baseline_preds):.1%}")
print(f"Model accuracy:    {accuracy_score(y_test, model_preds):.1%}")

# Rows are what actually happened, columns are what the model predicted.
cm = confusion_matrix(y_test, model_preds)
pd.DataFrame(cm,
             index=["Actual: stayed", "Actual: churned"],
             columns=["Predicted: stay", "Predicted: churn"])

In [ ]:
# TODO: pull the four counts (tn, fp, fn, tp) out of cm, then compute
# precision and recall from them with plain arithmetic.
# Do NOT use a package function (precision_score, recall_score,
# classification_report, ...) -- the self-check below uses those to grade you.
# YOUR CODE HERE

print(f"Precision: {precision:.1%}   Recall: {recall:.1%}")

In [ ]:
from sklearn.metrics import precision_score, recall_score

check_equal(int(tn + fp + fn + tp), len(y_test), "the four counts cover every test customer")
check_equal(int(tp), int(((model_preds == 1) & (y_test == 1)).sum()), "tp counts flagged customers who actually churned")
check_close(precision, precision_score(y_test, model_preds), "precision matches its definition")
check_close(recall, recall_score(y_test, model_preds), "recall matches its definition")
check_equal(accuracy_score(y_test, model_preds) > accuracy_score(y_test, baseline_preds), True, "model beats the baseline")

## 5. Interpret — Part B: Decide

A retention call costs Customer Success agent time and system overhead —
call it **\$40** per outreach. The value of a save is grounded in the data
rather than guessed: 88.5% of churners here are on **month-to-month**
contracts, not locked in, so a "save" buys a bounded number of extra months
rather than a lifetime. Assume a save is worth **6 months of the churned
customer's own monthly charges**.

*Using your own model's predicted probabilities on the test set (`model_proba`),
sweep a grid of thresholds and pick the one that maximizes expected net
benefit:* `(saves x save_value) - (calls x call_cost)`.

In [ ]:
call_cost = 40  # assumed: ~15 min of agent time + system/incentive overhead
save_value = 6 * df.loc[df["Churn"] == 1, "MonthlyCharges"].median()  # 6 months, at the typical churner's rate
print(f"call_cost=${call_cost}   save_value=${save_value:.0f}")

In [ ]:
# TODO: sweep thresholds against call_cost/save_value using model_proba;
# pick the threshold that maximizes expected net benefit.
# YOUR CODE HERE

print(f"Chosen threshold: {best_threshold:.2f}  ->  net benefit: ${best_net_benefit:,.0f}")

In [ ]:
check_in_range(best_threshold, 0.05, 0.95, "chosen threshold falls inside the swept range")
check_equal(best_net_benefit >= 0, True, "net benefit was actually computed from the sweep")

**Your answer:** *\<the threshold you chose\>*

**Justification (3–5 sentences):** _Your answer here._ 

## 6. Recommend

*Not graded this week.* Lesson 11's own recommendation stopped at "route
the top-risk decile to retention" — a modeling call, not a business one.
Your Part B answer above is closer to the real question, but writing it up
for a stakeholder — what to do Monday, what would change the
recommendation, what its limits are — is Part C, and that starts next week.

Hang onto your Part B number: you'll make this same call again in Week 5,
on a stronger model, and it'll be worth comparing whether a better model
actually changes the business answer.

## Rubric

*This week: three of the four criteria — Communication and the second half
of Justification (Part C's) aren't graded until Week 3, when Part C
exists. 70 pts total.*

| Criterion | Points | What it measures |
| --- | --- | --- |
| **Correctness** | 40 | Part A's checkpoints pass — the pipeline runs and produces the right shapes/values. |
| **Decision Quality** | 20 | Part B's threshold falls in a defensible range given the stated cost asymmetry and *your own* model's results. |
| **Justification** | 10 | Part B's reasoning engages with your own numbers. |